In [5]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import os

DATA_PATH = './../../data/housing/processed/price_paid_model_ready.parquet'
MODEL_PATH = './../../data/trained_models/housing/'
os.makedirs(MODEL_PATH, exist_ok=True)

In [6]:
# Cell 2: Load data
df = pd.read_parquet(DATA_PATH)

print("Rows:", len(df))
df.head()

Rows: 22480822


,price,sale_date,property_type,old_new,duration,town_city,district,county,record_status___monthly_file_only,sale_year
0,25000,1995-08-18,T,N,Freehold,OLDHAM,OLDHAM,GREATER MANCHESTER,A,1995
1,42500,1995-08-09,S,N,Freehold,GRAYS,THURROCK,THURROCK,A,1995
2,45000,1995-06-30,T,N,Freehold,HIGHBRIDGE,SEDGEMOOR,SOMERSET,A,1995
3,43150,1995-11-24,T,N,Freehold,BEDFORD,NORTH BEDFORDSHIRE,BEDFORDSHIRE,A,1995
4,18899,1995-06-23,S,N,Freehold,WAKEFIELD,LEEDS,WEST YORKSHIRE,A,1995


In [7]:
# Cell 3: Convertimos sale_date y creamos las columnas temporales
df['sale_date'] = pd.to_datetime(df['sale_date'])

df['Year'] = df['sale_date'].dt.year
df['Month'] = df['sale_date'].dt.month
df['Day'] = df['sale_date'].dt.day
df['Weekday'] = df['sale_date'].dt.weekday

# Creamos df_model igual que PyCaret (quitamos sale_date)
df_model = df.drop(columns=['sale_date'])

In [8]:
# Cell 4: Feature selection idéntica a PyCaret

target = 'price'

numeric_features = ['Year','Month','Day','Weekday']

categorical_features = [
    'property_type','old_new','duration',
    'town_city','district','county',
    'record_status___monthly_file_only'
]

feature_cols = numeric_features + categorical_features

X = df_model[feature_cols]
y = df_model[target]

In [9]:
# Cell 5: Conversión a categorías para LightGBM
for col in categorical_features:
    X[col] = X[col].astype('category')

/var/folders/6z/vs2gqd6s6zx7b6vqyq8nbwrh0000gn/T/ipykernel_2744/1775710158.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype('category')


In [10]:
# Cell 6: Downsample opcional (RECOMENDADO)
MAX_ROWS = 14_000_000  # cámbialo si quieres

if len(X) > MAX_ROWS:
    X_sample = X.sample(MAX_ROWS, random_state=123)
    y_sample = y.loc[X_sample.index]
else:
    X_sample = X
    y_sample = y

len(X_sample)

14000000

In [11]:
# Cell 7: División train/test respetando orden temporal
train_size = int(len(X_sample) * 0.9)

X_train = X_sample.iloc[:train_size]
y_train = y_sample.iloc[:train_size]

X_test  = X_sample.iloc[train_size:]
y_test  = y_sample.iloc[train_size:]

In [12]:
# Cell 8: LightGBM Dataset
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_features)
test_data  = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_features)

In [16]:
# Cell 9: Entrenar LightGBM
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 3,
    'verbosity': -1
}

lgb_model = lgb.train(
    params,
    train_data,
    num_boost_round=3000,
    valid_sets=[train_data, test_data],  # validación
    valid_names=['train','valid'],
)

In [18]:
# Cell 10: Evaluación
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = lgb_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

MAE: 54909.94826564501
RMSE: 182621.5945036423
R2: 0.4556946943384603


In [19]:
# Cell 11: Guardar modelo entrenado con joblib
import joblib

model_path = os.path.join(MODEL_PATH, "lightgbm_housing_model.pkl")
joblib.dump(lgb_model, model_path)

model_path

'./../../data/trained_models/housing/lightgbm_housing_model.pkl'

In [33]:
# Cell 12: Predicción con datos futuros
future_sample = pd.DataFrame([{
    'property_type': 'D',
    'old_new': 'Y',
    'duration': 'F',
    'town_city': 'BRIGHTON',
    'district': 'BRIGHTON AND HOVE',
    'county': 'EAST SUSSEX',
    'record_status___monthly_file_only': 'B',
    'Year': 2029,
    'Month': 5,
    'Day': 20,
    'Weekday': 2
}])

# categorías
for col in categorical_features:
    future_sample[col] = future_sample[col].astype('category')

predicted_price = lgb_model.predict(future_sample)
predicted_price

array([216384.56305041])